## SALARY SENSE PROJECT!!

# "SALARY PREDICTOR" Capstone Project

# Goal

Train a fine-tuned LLM that can estimate the expected salary of a job posting.

# Order of play

1. Step 1 — Dataset Exploration
2. Step 2 — Data Curation
3. Step 3 — LLM Extraction / Preprocessing
4. Step 4 — Prompt Generation
5. Step 5 — Dataset Preparation (JSONL)
6. Step 6 — Fine-Tuning Frontier Model
7. Step 7 — Evaluation
8. Step 8 — Model Comparison



Today we'll scrub our dataset from hugginface

The dataset is here:  
https://huggingface.co/datasets/princekhunt19/2025-ai-data-jobs-dataset

And the folder with all the product datasets is here:  
https://huggingface.co/datasets/princekhunt19/2025-ai-data-jobs-dataset/tree/main/data

# Step 3 — LLM Extraction / Preprocessing
In this step we extract structured information such as **experience** and **skills** from the job description using an LLM.

## Import all necessary libraries/functions

In [ ]:
import re
import sys
import json
import os
import time
import pandas as pd
from tqdm import tqdm
from src.extractor import extract_fields
from src.job_item import JobItem


## Load cleaned dataset
This assumes Step 2 has already produced the cleaned dataset.

In [ ]:

sys.path.append(os.path.abspath(".."))

with open("../data/processed/jobs_curated.json") as f:
    data = json.load(f)

print("Dataset size:", len(data))

items = []

for row in data:

    item = JobItem(
        title=row.get("positionName", ""),
        company=row.get("company", ""),
        location=row.get("location", ""),
        experience="Not specified",
        skills="",
        salary=row.get("salary_parsed", 0),
        description=row.get("description", "")
    )

    items.append(item)

print("Items created:", len(items))

## Inspect one job description
We will use this text to test the extraction pipeline.

In [ ]:
items[0].description[:500]

## Verify your api key

In [ ]:

print(os.getenv("OPENAI_API_KEY")[:10])

## Run extraction on a single example

In [ ]:
sample_description = items[0].description

result = extract_fields(sample_description)

print(result)

## Parse the extracted text
The extractor returns text, so we convert it to structured fields.

In [ ]:
def parse_extraction(text):

    data = {}

    for line in text.split("\n"):
        if ":" in line:
            key, value = line.split(":", 1)
            data[key.strip().lower()] = value.strip()

    return data

## Test parsing

In [ ]:
parsed = parse_extraction(result)

parsed

## Normalization of experience

In [ ]:


def normalize_experience(text):

    if not text:
        return "Not specified"

    text = text.lower()

    match = re.search(r"\d+\s*-\s*\d+\s*years|\d+\+?\s*years|\d+\s*years", text)

    if match:
        return match.group()

    return "Not specified"

## Apply extraction to a small batch
We start with a small batch to avoid large API costs.

In [ ]:
for item in tqdm(items):

    extracted = extract_fields(item.description)

    parsed = parse_extraction(extracted)

    item.skills = parsed.get("skills", "")

    exp = parsed.get("experience", "").strip()
    exp = normalize_experience(exp)

    item.experience = exp

    time.sleep(0.3)

## Inspect updated dataset

In [ ]:


df_items = pd.DataFrame([i.model_dump() for i in items[:20]])

df_items.head()

At this stage we now have:

- Job Title
- Location
- Skills (extracted)
- Experience (extracted)
- Salary

This structured dataset will be used in the next step to generate training prompts.

## One last test before saving

In [ ]:
df_items.experience.value_counts().head()

##  Save the cleaned dataset in the folder data/processed

In [ ]:


os.makedirs("../data/processed", exist_ok=True)

with open("../data/processed/jobs_cleaned.json", "w") as f:
    json.dump([i.model_dump() for i in items], f)

print("Clean dataset saved")